# TravelMind Notebook 2: Agentic Patterns
### IBS Agentic AI Practitioner Bootcamp | Day 6 | Advanced Strands

Notebook 1 kept **you** in charge of the path (chaining, routing, parallelization). This notebook hands part of the path to the **model**. Two patterns, one evolving airline agent, the same three dials measured every time.

**You will build:**
- **v5 Orchestrator-Workers** (agents as tools): a lead agent that decides *at runtime* which specialists to consult.
- **v6 Evaluator-Optimizer**: a generate then critique then revise loop that lifts customer-facing quality.

**The one question, still:** *who controls the path?* Watch it shift from you to the model as we climb.

## You are here

```mermaid
flowchart LR
    subgraph N1 [Notebook 1: workflows - you control the path]
        v1[v1 augmented] --> v2[v2 chaining] --> v3[v3 routing] --> v4[v4 parallel]
    end
    subgraph N2 [Notebook 2: agentic - the model controls more]
        v5[v5 orchestrator-workers] --> v6[v6 evaluator-optimizer]
    end
    subgraph N3 [Notebook 3]
        nx[swarm, graph, composition]
    end
    v4 --> v5
    v6 --> nx
```

The jump from N1 to N2 is the jump from a fixed flowchart to a model that picks steps. It buys flexibility and costs predictability. Cross it on purpose.

## The line we are crossing

| | Workflow (Notebook 1) | Agentic (this notebook) |
|---|---|---|
| Who picks the steps | You, in code | The model, at runtime |
| Can you draw the full flowchart up front | Yes | No, it depends on the input |
| Cost | Low, fixed | Higher, variable |
| Debuggability | Easy | Harder (path changes per run) |
| Reach for it when | The path is known | The path cannot be known until the model sees the input |

Keep this table in the corner of your eye for the rest of the notebook.

## How to run

**VS Code**
1. Open this `.ipynb`, pick a Python 3.10+ kernel.
2. Make AWS credentials available to boto3 (SSO or role session, or `aws configure`), with Bedrock access to Claude Haiku 4.5 in `us-east-1`.
3. Run cells top to bottom. First cell installs dependencies.

**Google Colab**
1. Upload, run the install cell.
2. Set session credentials: `import os; os.environ["AWS_ACCESS_KEY_ID"]=...` etc. plus `os.environ["AWS_REGION"]="us-east-1"`.
3. Run top to bottom.

This notebook is self-contained: it rebuilds the mock airline layer and the measurement harness from Notebook 1, so you can run it on its own.

In [ ]:
%pip install -q strands-agents strands-agents-tools nest_asyncio matplotlib

## Setup carried from Notebook 1

Three cells you have seen: model config, the mock airline operations layer, and the measurement harness. The harness gains one helper (`metered_multi`) so it can read token usage from **multi-agent** results (a Graph or Swarm), not just single agents. Run these, then we build.

In [ ]:
import os, time, json, asyncio, contextlib
import nest_asyncio
nest_asyncio.apply()  # allow asyncio.run(...) and orchestration inside notebook cells

from strands import Agent, tool
from strands.models import BedrockModel

REGION   = os.environ.get("AWS_REGION", "us-east-1")
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. => cross-region inference profile

haiku    = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
haiku_t0 = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0)  # for deterministic critics

print("Region:", REGION, "| Model:", HAIKU_ID)

In [ ]:
# --- Mock airline operations layer (deterministic fake data; no reservation system needed) ---
_PNRS = {
    "JX48Q2": {
        "surname": "Rao", "passenger_id": "P-100294", "loyalty_tier": "Gold",
        "segments": [
            {"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
             "status": "CANCELLED", "fare_basis": "TA21R"},
            {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
             "status": "ON TIME", "fare_basis": "TA21R"},
        ],
        "ancillaries": {"seat": "14C (paid)", "bags": 1},
    }
}
_FARE_RULES = {
    "TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
              "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."},
}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname (identity check).

    Args:
        record_locator: 6-character PNR, e.g. 'JX48Q2'.
        surname: Passenger surname for verification.
    Returns:
        JSON string of the booking, or an error if not found / surname mismatch.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules (refundability, change fee) for a fare basis code.

    Args:
        fare_basis: Fare basis code, e.g. 'TA21R'.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights to re-accommodate a passenger after a disruption.

    Args:
        origin: Origin airport code, e.g. 'BLR'.
        destination: Destination airport code, e.g. 'BOM'.
        after: ISO datetime; only flights departing after this are returned.
    Returns:
        JSON string list of candidate flights.
    """
    options = [
        {"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
        {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9},
    ]
    return json.dumps(options)


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits for a passenger.

    Args:
        passenger_id: Internal passenger id, e.g. 'P-100294'.
    Returns:
        JSON string of tier and benefits.
    """
    benefits = {"Gold": ["priority rebooking", "waived change fee on same-day", "lounge access"]}
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold", "benefits": benefits["Gold"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from the booking's fare rules and the stated reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason, e.g. 'flight cancelled by airline'.
    Returns:
        JSON string with an eligibility signal and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    fb = rec["segments"][0]["fare_basis"]
    rules = _FARE_RULES.get(fb, {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    eligible = airline_caused or rules.get("refundable", False)
    return json.dumps({"eligible": eligible, "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Tools ready: get_pnr, get_fare_rules, search_reaccommodation, get_loyalty, check_refund_eligibility")

In [ ]:
# --- Measurement harness (from NB1) + metered_multi for Graph/Swarm results ---
# Illustrative Bedrock list prices, USD per 1M tokens: (input, output). VERIFY before quoting.
PRICES  = {"haiku": (1.00, 5.00), "sonnet": (3.00, 15.00)}
_LEDGER = []   # (tier, {"input","output"}, ncalls)
RESULTS = {}   # label -> dials


def usage_of(result) -> dict:
    """Token usage from a Strands result (single-agent .metrics.accumulated_usage or multi-agent .accumulated_usage)."""
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None)
    if u is None:
        u = getattr(result, "accumulated_usage", None) or {}
    def g(*keys):
        for k in keys:
            if k in u:
                return u[k]
        return 0
    return {"input": g("inputTokens", "input_tokens"), "output": g("outputTokens", "output_tokens")}


def _nodes_run(result):
    """Ordered node ids that executed (Graph .execution_order or Swarm .node_history)."""
    seq = getattr(result, "execution_order", None) or getattr(result, "node_history", None) or []
    out = []
    for n in seq:
        out.append(getattr(n, "node_id", None) or getattr(n, "id", None) or str(n))
    return out


def final_text(result):
    """Best-effort final answer text from a multi-agent result (last node that ran)."""
    ids = _nodes_run(result)
    if ids:
        try:
            return str(result.results[ids[-1]].result)
        except Exception:
            pass
    return str(result)


def _record(res, tier="haiku"):
    _LEDGER.append((tier, usage_of(res), 1))


def metered(agent, prompt, tier="haiku"):
    """Run a single agent, record usage, return text."""
    res = agent(prompt)
    _record(res, tier)
    return str(res)


def metered_multi(orchestrator, prompt, tier="haiku"):
    """Run a Graph/Swarm, record its accumulated usage, return (final_text, raw_result)."""
    res = orchestrator(prompt)
    ncalls = max(1, len(_nodes_run(res)))
    _LEDGER.append((tier, usage_of(res), ncalls))
    return final_text(res), res


@contextlib.contextmanager
def meter(label):
    """Time a block; sum cost + token + call count across every recorded call inside it."""
    _LEDGER.clear()
    t0 = time.time()
    try:
        yield
    finally:
        latency = time.time() - t0
        cost = tin = tout = ncalls = 0
        for tier, u, c in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"] / 1e6 * pin + u["output"] / 1e6 * pout
            tin += u["input"]; tout += u["output"]; ncalls += c
        RESULTS[label] = {"latency_s": round(latency, 2), "input_tokens": tin,
                          "output_tokens": tout, "calls": ncalls, "cost_usd": round(cost, 6)}
        r = RESULTS[label]
        print(f"[{label}]  calls/nodes={r['calls']}  latency={r['latency_s']}s  "
              f"tokens={tin}in+{tout}out  cost=${r['cost_usd']:.6f}")

## v5: Orchestrator-Workers (Agents as Tools)

A lead agent talks to the customer, then decides **at runtime** which specialists to call and in what order. Each specialist is an agent wrapped as a tool. It mirrors a human team: the lead delegates as the problem unfolds.

| Concept card | |
|---|---|
| What it is | A manager agent that calls specialist agents as tools |
| Who controls the path | The model (it chooses which specialists, and how many) |
| The move | Wrap each specialist as a tool; give the orchestrator the toolbox |
| Cost shape | Higher and variable (every consulted specialist is a call) |

```mermaid
flowchart TD
    U([Customer]) --> O[Orchestrator]
    O -->|calls as needed| S1[flight_specialist]
    O -->|calls as needed| S2[fare_specialist]
    O -->|calls as needed| S3[refund_specialist]
    O -->|calls as needed| S4[loyalty_specialist]
    S1 --> O
    S2 --> O
    S3 --> O
    S4 --> O
    O --> R([One synthesized reply])
```

### Connect the dots: routing (v3) vs orchestrator-workers (v5)

Same tree shape on the whiteboard, completely different control. This is the distinction people blur, and it costs them.

```mermaid
flowchart LR
    subgraph R [v3 Routing: you pick ONE branch]
        m1([msg]) --> c1{classifier} -->|one label| b1[one specialist]
    end
    subgraph O [v5 Orchestrator: model picks WHICH and HOW MANY]
        m2([msg]) --> o2[orchestrator] --> b2[specialist A]
        o2 --> b3[specialist B]
        o2 --> b4[specialist C]
    end
```

| | v3 Routing | v5 Orchestrator-Workers |
|---|---|---|
| Who decides the branch | Your code, from a fixed label | The model, at runtime |
| How many branches run | Exactly one | Zero to many, model's call |
| Best for | Clean, separable categories | Messy multi-part queries you cannot pre-slice |
| Determinism | High | Medium |
| Cost | Lowest per query | Higher, variable |

**Rule of thumb:** if you can write the `if/elif` for the branch, use routing. If you cannot, that uncertainty is exactly what the orchestrator is for.

### Build it (the @tool wrapper variant)

Strands gives three ways to make an agent a tool: pass it directly in `tools=[...]`, use `.as_tool(...)`, or wrap it in `@tool` for full control. We use the **wrapper** here for one practical reason: it lets us record each specialist's token usage into the same meter, so the cost dial reflects the *whole* delegation, not just the orchestrator's own tokens.

In [ ]:
# Each specialist is a focused sub-agent, wrapped as a tool. The wrapper records the
# sub-agent's usage into the active meter so v5 cost includes specialist tokens.

@tool
def flight_specialist(query: str) -> str:
    """Report flight status and segment details for a PNR. Verify identity (record locator + surname) first."""
    a = Agent(model=haiku, name="flight_specialist",
              system_prompt="Report flight status and segment details for a PNR. Verify identity first. Use tools; never invent facts.",
              tools=[get_pnr])
    res = a(query); _record(res); return str(res)


@tool
def fare_specialist(query: str) -> str:
    """Explain fare rules, change fees, and whether a change is involuntary (fees waived)."""
    a = Agent(model=haiku, name="fare_specialist",
              system_prompt="Explain fare rules, change fees, and whether a change is airline-caused (involuntary, fees waived). Use tools.",
              tools=[get_pnr, get_fare_rules])
    res = a(query); _record(res); return str(res)


@tool
def refund_specialist(query: str) -> str:
    """Decide refund eligibility strictly from fare rules and the disruption reason. Never promise a refund without checking."""
    a = Agent(model=haiku, name="refund_specialist",
              system_prompt="Decide refund eligibility strictly from fare rules and the disruption reason. Never promise a refund without checking eligibility.",
              tools=[get_pnr, check_refund_eligibility])
    res = a(query); _record(res); return str(res)


@tool
def loyalty_specialist(query: str) -> str:
    """Report loyalty tier and the benefits relevant to a disruption (priority rebooking, fee waivers)."""
    a = Agent(model=haiku, name="loyalty_specialist",
              system_prompt="Report loyalty tier and the benefits relevant to a disruption (priority rebooking, waivers). Use tools.",
              tools=[get_pnr, get_loyalty])
    res = a(query); _record(res); return str(res)

ORCH_PROMPT = (
    "You are TravelMind's support lead. Read the customer's message, then consult ONLY the "
    "specialists needed to cover every part of it. Your specialist tools are: flight_specialist, "
    "fare_specialist, refund_specialist, loyalty_specialist. Call them as needed, then synthesize "
    "ONE clear, correct, warm reply. Verify identity (record locator + surname) before sharing details. "
    "Never invent flights, fares, or policy."
)
orchestrator = Agent(model=haiku, name="orchestrator", system_prompt=ORCH_PROMPT,
                     tools=[flight_specialist, fare_specialist, refund_specialist, loyalty_specialist])

print("Orchestrator wired with 4 specialist tools.")

In [ ]:
# A deliberately messy, multi-part query: no clean single branch fits it.
messy = ("Hi, Rao here, PNR JX48Q2. My BLR-DEL flight got cancelled. I'm Gold and I paid for seat 14C. "
         "Can I get on the next flight to BOM today, and either a refund or a fee waiver? "
         "And what happens to my paid seat?")

with meter("v5_orchestrator_workers"):
    reply = metered(orchestrator, messy)
    print(reply)

### Read the run as a sequence

The orchestrator decided this path; you did not script it. On a different query it would call a different subset.

```mermaid
sequenceDiagram
    participant C as Customer
    participant O as Orchestrator
    participant F as flight_specialist
    participant Fa as fare_specialist
    participant R as refund_specialist
    participant L as loyalty_specialist
    C->>O: cancelled flight + Gold + paid seat + refund-or-waiver (one message)
    O->>F: what are my options to BOM today?
    O->>Fa: is this involuntary? fees waived?
    O->>R: refund eligible given cancellation?
    O->>L: what Gold benefits apply?
    O->>C: one synthesized reply covering every part
```

### The cost of delegation: v5 vs one generalist

Delegation is not free. To see what it buys and what it costs, run the **same** messy query through a single generalist agent that carries every tool, then compare the dials.

In [ ]:
# Baseline: one generalist agent holding all tools, answering the same messy query.
generalist = Agent(model=haiku, name="generalist",
    system_prompt="You are TravelMind support. Verify identity, use tools, never invent facts. Handle status, changes, refunds, loyalty in one reply.",
    tools=[get_pnr, get_fare_rules, search_reaccommodation, get_loyalty, check_refund_eligibility])

with meter("v5_baseline_generalist"):
    print(metered(generalist, messy))

In [ ]:
import matplotlib.pyplot as plt

pair = [k for k in ["v5_baseline_generalist", "v5_orchestrator_workers"] if k in RESULTS]
lat  = [RESULTS[k]["latency_s"] for k in pair]
cost = [RESULTS[k]["cost_usd"] * 1000 for k in pair]
toks = [RESULTS[k]["input_tokens"] + RESULTS[k]["output_tokens"] for k in pair]
nice = {"v5_baseline_generalist": "one generalist", "v5_orchestrator_workers": "orchestrator + specialists"}
names = [nice[k] for k in pair]

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
ax[0].bar(names, lat,  color=["#8aa"]*len(pair)); ax[0].set_title("Latency (s) - lower better")
ax[1].bar(names, cost, color=["#a8a"]*len(pair)); ax[1].set_title("Cost (USD x1000) - lower better")
ax[2].bar(names, toks, color=["#aa8"]*len(pair)); ax[2].set_title("Total tokens - lower better")
for a in ax: a.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

print(f"{'version':<28}{'calls/nodes':>12}{'latency_s':>11}{'tokens':>9}{'cost_usd':>12}")
for k in pair:
    r = RESULTS[k]
    print(f"{k:<28}{r['calls']:>12}{r['latency_s']:>11}{r['input_tokens']+r['output_tokens']:>9}{r['cost_usd']:>12.6f}")

**How to read it.** The orchestrator usually spends **more** tokens and money than one generalist: it pays for the lead's reasoning *plus* each specialist it consults. What you buy is cleaner separation of concerns, focused prompts, and specialists you can test and swap in isolation. On a simple query that a generalist nails, this is waste. On a genuinely messy query, the structure earns its cost in reliability.

**Accounting note.** When agents are used as tools, the orchestrator's own `accumulated_usage` may not include the sub-agents' tokens. The `@tool` wrapper here records each specialist call into the meter, so this cost is honest. In production, capture per-agent metrics or OpenTelemetry traces rather than reading one top-level number.

> **What changes in production**
> - Route the orchestrator and cheap specialists to Haiku; promote only a genuinely hard specialist to Sonnet.
> - Put a timeout on the orchestrator and every specialist so one slow tool cannot stall the reply.
> - Log which specialists fired per query; it is your bill and your audit trail.

## v6: Evaluator-Optimizer

One agent generates, a second critiques against **explicit written criteria**, the generator revises. Loop until the critic approves or you hit a cap. This is how you reach customer-facing quality without a human reading every reply.

| Concept card | |
|---|---|
| What it is | Generate then critique then revise, in a capped loop |
| Who controls the path | Shared: you own the loop, the model owns the content |
| The move | A critic agent with a rubric + a stop condition |
| Cost shape | Higher, grows per iteration (so you cap it) |

```mermaid
flowchart TD
    In([Task]) --> G[Generator: draft reply]
    G --> C{Critic: policy + tone}
    C -->|revision needed + feedback| G
    C -->|approved| Out([Send])
```

We build it **twice**: first as a plain Python loop so the mechanics are obvious, then as a Strands cyclic Graph so you see the framework-native form with caps and state reset built in.

### The critic's rubric (the part that actually matters)

A loop with vague criteria just burns tokens. TravelMind's critic checks a concrete standing policy. Written down, it becomes testable.

**TravelMind standing policy**
- Identity is verified (record locator + surname) before any booking detail is shared.
- No refund is promised without checking fare rules and the disruption reason.
- Airline-caused (involuntary) changes waive change fees; say so when it applies.
- A long delay or cancellation triggers duty-of-care language, not a brush-off.
- No other passenger's data is disclosed.
- Tone is warm, concise, and free of false promises.

The critic returns exactly one verdict line: `APPROVED` or `REVISION NEEDED: <what to fix>`. Those literal tokens are what the loop and the Graph branch on.

### v6a: the transparent loop

Read this first. It is just a `while` with a cap. Everything the framework does later, you can see here in plain Python.

In [ ]:
GEN_PROMPT = ("You are TravelMind. Draft a customer reply that resolves the request. "
              "Verify identity, use tools for facts, never invent flights, fares, or policy. Keep it warm and concise.")

POLICY = ("Standing policy: identity verified (record locator + surname) before sharing details; "
          "no refund promised without checking fare rules and reason; involuntary changes waive fees (say so); "
          "cancellation/long delay gets duty-of-care language; never disclose other passengers' data; warm, concise, no false promises.")

CRITIC_PROMPT = (POLICY + "\n\nYou are a strict reviewer. Given a draft reply, check it against the policy and tone. "
                 "Respond with a short critique, then end with EXACTLY one final line: "
                 "'APPROVED' if it fully complies, otherwise 'REVISION NEEDED: <specific fixes>'.")

generator = Agent(model=haiku, name="generator", system_prompt=GEN_PROMPT,
                  tools=[get_pnr, get_fare_rules, search_reaccommodation, get_loyalty, check_refund_eligibility])
critic    = Agent(model=haiku_t0, name="critic", system_prompt=CRITIC_PROMPT)  # temperature 0 for a stable rubric


def cost_of(res, tier="haiku"):
    u = usage_of(res); pin, pout = PRICES[tier]
    return u["input"] / 1e6 * pin + u["output"] / 1e6 * pout


def evaluate_optimize(task, max_passes=4):
    """Generate -> critique -> revise until APPROVED or max_passes. Records usage; returns (final_draft, history)."""
    history, feedback, draft = [], "", ""
    for i in range(1, max_passes + 1):
        prompt = task if not feedback else task + "\n\nRevise using this reviewer feedback:\n" + feedback
        gres = generator(prompt); _record(gres); draft = str(gres)
        cres = critic("Draft to review:\n" + draft); _record(cres); verdict = str(cres)
        low = verdict.lower()
        approved = ("approved" in low) and ("revision needed" not in low)
        history.append({"pass": i, "draft": draft, "verdict": verdict, "approved": approved,
                        "gen_cost": cost_of(gres), "critic_cost": cost_of(cres)})
        if approved:
            break
        feedback = verdict
    return draft, history

q_change = ("This is Rao, PNR JX48Q2. My first flight BLR-DEL was cancelled by the airline. "
            "I want to still reach BOM today and I want my change fee waived. Can you sort it?")

with meter("v6_evaluator_optimizer_loop"):
    final_draft, history = evaluate_optimize(q_change, max_passes=4)

print(f"passes run: {len(history)}  |  final verdict approved: {history[-1]['approved']}")
print("\n----- FINAL REPLY -----\n", final_draft)

### Before and after: what the critic changed

The loop's value is visible only if you look at the first draft next to the last. Run this to see the delta and how many passes it took.

In [ ]:
print(f"Total passes: {len(history)}\n")
first, last = history[0], history[-1]

print("FIRST DRAFT (pass 1):")
print(first["draft"][:700])
print("\nCRITIC VERDICT ON PASS 1:")
print(first["verdict"].strip().splitlines()[-1] if first["verdict"].strip() else "(empty)")

if len(history) > 1:
    print("\nFINAL DRAFT (pass %d):" % last["pass"])
    print(last["draft"][:700])
    print("\nFINAL CRITIC VERDICT:")
    print(last["verdict"].strip().splitlines()[-1])
else:
    print("\nThe first draft already passed the critic. The mechanism still ran; on a harder task it revises.")

In [ ]:
# Iteration cost curve: cost accrues every pass, so the cap is a budget lever, not a nicety.
import matplotlib.pyplot as plt

passes    = [h["pass"] for h in history]
per_pass  = [h["gen_cost"] + h["critic_cost"] for h in history]
cumulative = []
run = 0.0
for c in per_pass:
    run += c; cumulative.append(run * 1000)  # USD x1000

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar([str(p) for p in passes], [c * 1000 for c in per_pass], color="#a8a")
ax[0].set_title("Cost per pass (USD x1000)"); ax[0].set_xlabel("pass")
ax[1].plot([str(p) for p in passes], cumulative, marker="o", color="#733")
ax[1].set_title("Cumulative cost (USD x1000)"); ax[1].set_xlabel("pass")
plt.tight_layout(); plt.show()

print("Each pass is one generator call plus one critic call. Uncapped, this line never stops climbing.")

### v6b: the same loop as a Strands cyclic Graph

The transparent loop taught the idea. In production you want the framework to own the cap, the state reset, and the branch conditions. That is a **cyclic Graph**: a draft node, a critic node, a conditional edge back to draft on failure, and a conditional edge to publish on approval.

```mermaid
flowchart TD
    D[draft node] --> R{critic node}
    R -->|revision needed| D
    R -->|approved| P[publish node]
```

Two things the Graph handles for you that the loop did by hand:
- `set_max_node_executions(...)` caps the whole thing (your hard stop).
- `reset_on_revisit(True)` gives the draft node a clean slate each pass; the critic's feedback still reaches it through **input propagation** (the critic node feeds the draft node on the revise edge).

In [ ]:
from strands.multiagent import GraphBuilder

# Nodes are agents. The critic MUST emit the literal tokens the edges branch on.
draft_node = Agent(model=haiku, name="draft_node", system_prompt=GEN_PROMPT,
                   tools=[get_pnr, get_fare_rules, search_reaccommodation, get_loyalty, check_refund_eligibility])
critic_node = Agent(model=haiku_t0, name="critic_node", system_prompt=CRITIC_PROMPT)
publish_node = Agent(model=haiku, name="publish_node",
                     system_prompt="Format the approved reply for sending. Do not change its meaning; fix only clarity and spacing.")


# Conditional edges read the critic node's latest output (canonical Strands feedback-loop pattern).
def needs_revision(state):
    r = state.results.get("critic_node")
    if not r:
        return False
    return "revision needed" in str(r.result).lower()


def is_approved(state):
    r = state.results.get("critic_node")
    if not r:
        return False
    text = str(r.result).lower()
    return ("approved" in text) and ("revision needed" not in text)


builder = GraphBuilder()
builder.add_node(draft_node,   "draft_node")
builder.add_node(critic_node,  "critic_node")
builder.add_node(publish_node, "publish_node")

builder.add_edge("draft_node", "critic_node")
builder.add_edge("critic_node", "draft_node",   condition=needs_revision)   # feedback loop
builder.add_edge("critic_node", "publish_node", condition=is_approved)

builder.set_entry_point("draft_node")
builder.set_max_node_executions(8)   # hard stop for the loop
builder.set_execution_timeout(300)
builder.reset_on_revisit(True)       # fresh draft state each pass; critic feedback still propagates in

eval_opt_graph = builder.build()
print("Cyclic evaluator-optimizer graph built.")

In [ ]:
with meter("v6_evaluator_optimizer_graph"):
    reply, res = metered_multi(eval_opt_graph, q_change)

print("execution order:", _nodes_run(res))
print("\n----- GRAPH FINAL REPLY -----\n", reply)

**v6a vs v6b, side by side in your head:**

| | v6a hand-rolled loop | v6b cyclic Graph |
|---|---|---|
| Cap | Your `range(max_passes)` | `set_max_node_executions` |
| State reset | You rebuilt the prompt | `reset_on_revisit(True)` |
| Branch on verdict | Your `if approved` | Conditional edges |
| When to use | Learning, tiny scripts, full control | Production, observability, reuse |

Same pattern, same dials. The Graph is the version you ship.

> **What changes in production**
> - Cap every loop. An uncapped evaluator is an unbounded bill; this is the single most common agentic cost incident.
> - Cache the critic's system prompt (it never changes) once the token floor is met.
> - Keep the critic at temperature 0 so the rubric is stable across runs.

## Recap and bridge to Notebook 3

You handed the path to the model, twice:
- **v5 Orchestrator-Workers**: the model chooses which specialists to consult. Flexible on messy queries, higher and variable cost.
- **v6 Evaluator-Optimizer**: a critic loop lifts customer-facing quality. The biggest single quality lever, and the biggest loop-cost risk (so you cap it).

The control axis kept moving:

```mermaid
flowchart LR
    A[v1-v4 workflows: you own the path] --> B[v5 orchestrator: model picks specialists] --> C[v6 evaluator: model refines, you own the loop]
    C --> D[Notebook 3: swarm - peers own the path; graph - you own the structure]
```

**Notebook 3** takes this to the two extremes: a **Swarm** where peer agents hand off with no boss (the model owns almost everything), and a **Graph** where you define a deterministic, auditable structure (you own it again, with the model reasoning inside nodes). Then we **compose** them: a compliance graph with a creative swarm living inside one node. Same harness, same TravelMind, so every dial stays comparable.